# Exploratory Data Analysis
## Heat-Stress Early Warning for Baghdad

This notebook provides interactive exploration of the hourly weather dataset,
heat-stress labels, NHRI, and feature distributions.
For the full automated pipeline, run `python main.py`.

In [ ]:
import os, sys
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
print('Imports OK')

## 1. Load Raw Data

In [ ]:
from src.data_loader import load_raw_data
from src.utils import create_directories
create_directories()
locations_df, hourly_df, daily_df = load_raw_data()
print(hourly_df.dtypes)

## 2. Preprocess Hourly Data

In [ ]:
from src.preprocessing import preprocess_hourly_weather
clean_df = preprocess_hourly_weather(hourly_df)
clean_df.head()

## 3. Heat-Stress Labels

In [ ]:
from src.labeling import create_heat_stress_labels
labeled_df = create_heat_stress_labels(clean_df)

# Class distribution
dist = labeled_df['heat_risk_class'].value_counts().sort_index()
labels = ['Normal', 'Caution', 'Danger', 'Extreme Danger']
colors = ['#4CAF50', '#FFC107', '#FF5722', '#B71C1C']

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(labels, dist.values, color=colors)
ax.set_title('Heat-Stress Class Distribution', fontweight='bold')
ax.set_ylabel('Count')
for i, v in enumerate(dist.values):
    ax.text(i, v + 100, f'{v:,}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

## 4. Feature Engineering

In [ ]:
from src.feature_engineering import engineer_features
feat_df = engineer_features(labeled_df)
print(f'Shape after feature engineering: {feat_df.shape}')
feat_df.describe()

## 5. Nighttime Heat Recovery Index (NHRI)

In [ ]:
from src.nhri import compute_nhri, merge_nhri_into_hourly
daily_nhri = compute_nhri(feat_df)
full_df = merge_nhri_into_hourly(feat_df, daily_nhri)

# Yearly NHRI_35 trend
full_df['year'] = pd.to_datetime(full_df['datetime']).dt.year
yearly_nhri = full_df.groupby('year')['NHRI_35'].mean()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(yearly_nhri.index, yearly_nhri.values, marker='o', color='#7B1FA2', linewidth=2)
ax.fill_between(yearly_nhri.index, yearly_nhri.values, alpha=0.2, color='#7B1FA2')
ax.set_title('Yearly Mean NHRI₃₅', fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('NHRI₃₅')
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

daily_nhri.head()

## 6. Monthly FeelsLikeC

In [ ]:
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
monthly = full_df.groupby('month')['FeelsLikeC'].agg(['mean', 'max', 'min'])

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(monthly.index, monthly['mean'], marker='o', linewidth=2, color='#E53935', label='Mean')
ax.fill_between(monthly.index, monthly['min'], monthly['max'], alpha=0.2, color='#E53935', label='Min-Max range')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_names)
ax.axhline(52, color='#B71C1C', linestyle='--', linewidth=1.2, label='Extreme Danger (52°C)')
ax.axhline(40, color='#FF5722', linestyle='--', linewidth=1.2, label='Danger (40°C)')
ax.axhline(32, color='#FFC107', linestyle='--', linewidth=1.2, label='Caution (32°C)')
ax.set_title('Monthly FeelsLikeC Statistics', fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('FeelsLikeC (°C)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 7. Chronological Split

In [ ]:
from src.preprocessing import chronological_split
train_df, val_df, test_df = chronological_split(full_df)

print(f'Train: {len(train_df):,} rows')
print(f'Val  : {len(val_df):,} rows')
print(f'Test : {len(test_df):,} rows')

## 8. Correlation Heatmap

In [ ]:
key_cols = ['tempC', 'FeelsLikeC', 'HeatIndexC', 'DewPointC', 'WindChillC',
            'humidity', 'pressureMB', 'windspeedKmph', 'cloudcover', 'precipMM',
            'uvIndex', 'NHRI_35', 'NHRI_40', 'hour']
available = [c for c in key_cols if c in full_df.columns]
corr = full_df[available].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.5, annot_kws={'size': 8})
ax.set_title('Key Feature Correlation Heatmap', fontsize=13, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 9. Summary Statistics

In [ ]:
summary_cols = ['FeelsLikeC', 'tempC', 'HeatIndexC', 'humidity', 'windspeedKmph', 'NHRI_35', 'NHRI_40']
available = [c for c in summary_cols if c in full_df.columns]
full_df[available].describe().round(2)

## 10. Extreme Heat Events Timeline

In [ ]:
extreme = full_df[full_df['heat_risk_class'] == 3].copy()
extreme['year'] = pd.to_datetime(extreme['datetime']).dt.year
yearly_extreme = extreme.groupby('year').size()

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(yearly_extreme.index, yearly_extreme.values, color='#B71C1C')
ax.set_title('Annual Hours of Extreme Danger (FeelsLikeC ≥ 52°C)', fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Number of Hours')
plt.tight_layout()
plt.show()

print(f'Total Extreme Danger hours: {len(extreme):,}')
print(f'Maximum in a single year  : {yearly_extreme.max():,} ({yearly_extreme.idxmax()})')